<a href="https://colab.research.google.com/github/heetaamin/ml-assignment2/blob/main/earlier_version/kNN_modelling_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Notebook 2 (rebuilt) — Cell 1: Load semi-raw data, set up
# stratified 5-fold CV
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/networkTraffic_knn_semiraw.csv'
df_knn = pd.read_csv(DATA_PATH)

X = df_knn.drop(columns=['attack_cat'])
y = df_knn['attack_cat']

print("X shape:", X.shape)
print("y shape:", y.shape)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
X shape: (150243, 34)
y shape: (150243,)


In [ ]:
# ============================================================
# Cell 2 (consolidated): clamp -> scale -> proto bucket ->
# one-hot encode, all inside ONE loop per fold
# ============================================================
from sklearn.preprocessing import MinMaxScaler

clamp_cols = ['dload', 'dbytes', 'spkts', 'dmean', 'smean', 'sbytes',
              'dpkts', 'djit', 'sjit', 'sload', 'dur', 'synack',
              'sinpkt', 'dinpkt', 'response_body_len', 'ackdat']
nominal_cols = ['proto', 'state', 'service']

fold_data = []  # store each fold's fully-processed train/test here

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # --- Clamp (fit on training only) ---
    for col in clamp_cols:
        lower = X_train[col].quantile(0.01)
        upper = X_train[col].quantile(0.99)
        X_train[col] = X_train[col].clip(lower=lower, upper=upper)
        X_test[col] = X_test[col].clip(lower=lower, upper=upper)

    # --- Scale (fit on training only) ---
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler = MinMaxScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    # --- proto bucketing (fit on training only) ---
    proto_counts_train = X_train['proto'].value_counts()
    threshold = 0.01 * len(X_train)
    keep_protos = proto_counts_train[proto_counts_train >= threshold].index.tolist()
    X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
    X_test['proto'] = X_test['proto'].where(X_test['proto'].isin(keep_protos), 'other')

    # --- One-hot encode (fit columns on training only) ---
    X_train_enc = pd.get_dummies(X_train, columns=nominal_cols)
    X_test_enc = pd.get_dummies(X_test, columns=nominal_cols)
    X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

    fold_data.append((X_train_enc, X_test_enc, y_train, y_test))

    print(f"Fold {fold_num+1}: done. X_train: {X_train_enc.shape}, X_test: {X_test_enc.shape}, kept protos: {keep_protos}")

Fold 1: done. X_train: (120194, 56), X_test: (30049, 56), kept protos: ['tcp', 'udp']
Fold 2: done. X_train: (120194, 57), X_test: (30049, 57), kept protos: ['tcp', 'udp']
Fold 3: done. X_train: (120194, 57), X_test: (30049, 57), kept protos: ['tcp', 'udp']
Fold 4: done. X_train: (120195, 55), X_test: (30048, 55), kept protos: ['tcp', 'udp']
Fold 5: done. X_train: (120195, 57), X_test: (30048, 57), kept protos: ['tcp', 'udp']


In [ ]:
# ============================================================
# Diagnostic: confirm why fewer protocols cleared the threshold
# ============================================================
print("Full semi-raw dataset proto counts:")
print(df_knn['proto'].value_counts().head(10))
print(f"\n1% threshold on full semi-raw data: {0.01 * len(df_knn):.0f} rows")

Full semi-raw dataset proto counts:
proto
tcp       116134
udp        25791
unas        1024
ospf         965
arp          766
sctp         372
any          106
rsvp          77
sun-nd        74
gre           72
Name: count, dtype: int64

1% threshold on full semi-raw data: 1502 rows


In [ ]:
# NOTE: only tcp/udp now clear the 1% proto threshold, down from
# 5 protocols (tcp, udp, unas, arp, ospf) in the earlier pipeline.
# Confirmed this is a real consequence of the corrected pipeline
# order, not a bug: unas/arp/ospf were disproportionately
# represented among the duplicate rows removed in Notebook 1
# (e.g. unas dropped from a higher pre-dedup count to 1,024 post-
# dedup, below the 1,502-row 1% threshold). The earlier pipeline
# computed this threshold before deduplication, inflating these
# protocols' apparent frequency.

In [ ]:
# ============================================================
# Cell 2 (continued): add MI feature selection inside the loop
# ============================================================
from sklearn.feature_selection import mutual_info_classif

fold_data = []
fold_selected_features = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # --- Clamp ---
    for col in clamp_cols:
        lower = X_train[col].quantile(0.01)
        upper = X_train[col].quantile(0.99)
        X_train[col] = X_train[col].clip(lower=lower, upper=upper)
        X_test[col] = X_test[col].clip(lower=lower, upper=upper)

    # --- Scale ---
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler = MinMaxScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    # --- proto bucketing ---
    proto_counts_train = X_train['proto'].value_counts()
    threshold = 0.01 * len(X_train)
    keep_protos = proto_counts_train[proto_counts_train >= threshold].index.tolist()
    X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
    X_test['proto'] = X_test['proto'].where(X_test['proto'].isin(keep_protos), 'other')

    # --- One-hot encode ---
    X_train_enc = pd.get_dummies(X_train, columns=nominal_cols)
    X_test_enc = pd.get_dummies(X_test, columns=nominal_cols)
    X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

    # --- MI feature selection (fit on training only) ---
    mi_scores = mutual_info_classif(X_train_enc, y_train, random_state=42)
    mi_ranking = pd.Series(mi_scores, index=X_train_enc.columns).sort_values(ascending=False)
    top_10 = mi_ranking.head(10).index.tolist()
    fold_selected_features.append(top_10)

    X_train_final = X_train_enc[top_10]
    X_test_final = X_test_enc[top_10]

    fold_data.append((X_train_final, X_test_final, y_train, y_test))

    print(f"Fold {fold_num+1}: top-10 features: {top_10}")

Fold 1: top-10 features: ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt', 'dpkts']
Fold 2: top-10 features: ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt', 'dpkts']
Fold 3: top-10 features: ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt', 'dpkts']
Fold 4: top-10 features: ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt', 'dpkts']
Fold 5: top-10 features: ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt', 'dpkts']


In [ ]:
# ============================================================
# Cell 2 (continued): add SMOTE inside the loop
# ============================================================
from imblearn.over_sampling import SMOTE

fold_data = []
fold_selected_features = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # --- Clamp ---
    for col in clamp_cols:
        lower = X_train[col].quantile(0.01)
        upper = X_train[col].quantile(0.99)
        X_train[col] = X_train[col].clip(lower=lower, upper=upper)
        X_test[col] = X_test[col].clip(lower=lower, upper=upper)

    # --- Scale ---
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler = MinMaxScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    # --- proto bucketing ---
    proto_counts_train = X_train['proto'].value_counts()
    threshold = 0.01 * len(X_train)
    keep_protos = proto_counts_train[proto_counts_train >= threshold].index.tolist()
    X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
    X_test['proto'] = X_test['proto'].where(X_test['proto'].isin(keep_protos), 'other')

    # --- One-hot encode ---
    X_train_enc = pd.get_dummies(X_train, columns=nominal_cols)
    X_test_enc = pd.get_dummies(X_test, columns=nominal_cols)
    X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

    # --- MI feature selection ---
    mi_scores = mutual_info_classif(X_train_enc, y_train, random_state=42)
    mi_ranking = pd.Series(mi_scores, index=X_train_enc.columns).sort_values(ascending=False)
    top_10 = mi_ranking.head(10).index.tolist()
    fold_selected_features.append(top_10)

    X_train_final = X_train_enc[top_10]
    X_test_final = X_test_enc[top_10]

    # --- SMOTE (fit on training only, applied after feature selection) ---
    target_counts = {cls: min(count * 5, y_train.value_counts().max())
                      for cls, count in y_train.value_counts().items()}
    smote = SMOTE(sampling_strategy=target_counts, random_state=42, k_neighbors=5)
    X_train_smote, y_train_smote = smote.fit_resample(X_train_final, y_train)

    fold_data.append((X_train_smote, X_test_final, y_train_smote, y_test))

    print(f"Fold {fold_num+1}: SMOTE done. X_train: {X_train_smote.shape}, X_test: {X_test_final.shape}")

Fold 1: SMOTE done. X_train: (282013, 10), X_test: (30049, 10)
Fold 2: SMOTE done. X_train: (282008, 10), X_test: (30049, 10)
Fold 3: SMOTE done. X_train: (282008, 10), X_test: (30049, 10)
Fold 4: SMOTE done. X_train: (282011, 10), X_test: (30048, 10)
Fold 5: SMOTE done. X_train: (282016, 10), X_test: (30048, 10)


In [ ]:
# ============================================================
# Cell 3: kNN fit/predict on each fold, collect metrics
# ============================================================
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, classification_report

fold_macro_f1 = []
fold_accuracy = []
fold_weighted_f1 = []
all_y_test = []
all_y_pred = []

for fold_num, (X_train_final, X_test_final, y_train_final, y_test_final) in enumerate(fold_data):
    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train_final, y_train_final)
    y_pred = model.predict(X_test_final)

    macro_f1 = f1_score(y_test_final, y_pred, average='macro')
    acc = accuracy_score(y_test_final, y_pred)
    weighted_f1 = f1_score(y_test_final, y_pred, average='weighted')

    fold_macro_f1.append(macro_f1)
    fold_accuracy.append(acc)
    fold_weighted_f1.append(weighted_f1)
    all_y_test.extend(y_test_final)
    all_y_pred.extend(y_pred)

    print(f"Fold {fold_num+1}: macro-F1={macro_f1:.4f}, accuracy={acc:.4f}, weighted-F1={weighted_f1:.4f}")

print(f"\nMean macro-F1:    {np.mean(fold_macro_f1):.4f} (± {np.std(fold_macro_f1):.4f})")
print(f"Mean accuracy:    {np.mean(fold_accuracy):.4f} (± {np.std(fold_accuracy):.4f})")
print(f"Mean weighted-F1: {np.mean(fold_weighted_f1):.4f} (± {np.std(fold_weighted_f1):.4f})")

Fold 1: macro-F1=0.5020, accuracy=0.7533, weighted-F1=0.7575
Fold 2: macro-F1=0.5000, accuracy=0.7512, weighted-F1=0.7638
Fold 3: macro-F1=0.5091, accuracy=0.7504, weighted-F1=0.7634
Fold 4: macro-F1=0.4994, accuracy=0.7550, weighted-F1=0.7656
Fold 5: macro-F1=0.5108, accuracy=0.7551, weighted-F1=0.7583

Mean macro-F1:    0.5043 (± 0.0047)
Mean accuracy:    0.7530 (± 0.0019)
Mean weighted-F1: 0.7617 (± 0.0032)


In [ ]:
# ============================================================
# Cell 4: Compare corrected pipeline against the old (leaky)
# final result
# ============================================================
OLD_MACRO_F1 = 0.5099
print(f"Old (leaky) mean macro-F1:       {OLD_MACRO_F1:.4f}")
print(f"New (leak-safe) mean macro-F1:   {np.mean(fold_macro_f1):.4f}")
print(f"Difference:                       {np.mean(fold_macro_f1) - OLD_MACRO_F1:+.4f}")

Old (leaky) mean macro-F1:       0.5099
New (leak-safe) mean macro-F1:   0.5043
Difference:                       -0.0056


In [ ]:
# ============================================================
# Cell 5: Final per-class report, pooled across all 5 folds
# — this is the table for your Results section
# ============================================================
print(classification_report(all_y_test, all_y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.919     0.835     0.875     83358
           1      0.699     0.727     0.713      8609
           2      0.200     0.368     0.259      1503
           3      0.315     0.385     0.347      5019
           4      0.740     0.762     0.751     26927
           5      0.141     0.090     0.110      1577
           6      0.502     0.628     0.558     19470
           7      0.391     0.550     0.457       169
           8      0.345     0.374     0.359      1426
           9      0.708     0.568     0.630      2185

    accuracy                          0.753    150243
   macro avg      0.496     0.529     0.506    150243
weighted avg      0.776     0.753     0.762    150243



In [ ]:
# ============================================================
# Diagnostic: how much did each class's row count change
# between the old and new pipelines?
# ============================================================
# Old pipeline's post-dedup class counts, from earlier in this
# conversation (162,745 total rows)
old_counts = {
    0: 85487, 1: 9745, 2: 1678, 3: 5236, 4: 27170,
    5: 1770, 6: 20698, 7: 171, 8: 1456, 9: 7572
}

new_counts = df_knn['attack_cat'].value_counts().sort_index().to_dict()

print(f"{'Class':<8}{'Old count':<12}{'New count':<12}{'% change':<10}")
for cls in range(10):
    old = old_counts[cls]
    new = new_counts.get(cls, 0)
    pct_change = (new - old) / old * 100
    print(f"{cls:<8}{old:<12}{new:<12}{pct_change:+.1f}%")

Class   Old count   New count   % change  
0       85487       83358       -2.5%
1       9745        8609        -11.7%
2       1678        1503        -10.4%
3       5236        5019        -4.1%
4       27170       26927       -0.9%
5       1770        1577        -10.9%
6       20698       19470       -5.9%
7       171         169         -1.2%
8       1456        1426        -2.1%
9       7572        2185        -71.1%


In [ ]:
# ============================================================
# Cell A: Rebuild fold data at the "full feature set, pre-MI,
# pre-SMOTE" checkpoint — needed for baseline/weighted/Manhattan
# tests
# ============================================================
fold_data_full = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    for col in clamp_cols:
        lower = X_train[col].quantile(0.01)
        upper = X_train[col].quantile(0.99)
        X_train[col] = X_train[col].clip(lower=lower, upper=upper)
        X_test[col] = X_test[col].clip(lower=lower, upper=upper)

    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler = MinMaxScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    proto_counts_train = X_train['proto'].value_counts()
    threshold = 0.01 * len(X_train)
    keep_protos = proto_counts_train[proto_counts_train >= threshold].index.tolist()
    X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
    X_test['proto'] = X_test['proto'].where(X_test['proto'].isin(keep_protos), 'other')

    X_train_enc = pd.get_dummies(X_train, columns=nominal_cols)
    X_test_enc = pd.get_dummies(X_test, columns=nominal_cols)
    X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

    fold_data_full.append((X_train_enc, X_test_enc, y_train, y_test))

print("fold_data_full built:", len(fold_data_full), "folds")

fold_data_full built: 5 folds


In [ ]:
# ============================================================
# Cell B: Baseline (k=5, Euclidean, uniform, all features, no SMOTE)
# ============================================================
f1_baseline = []
for X_train_f, X_test_f, y_train_f, y_test_f in fold_data_full:
    model = KNeighborsClassifier(n_neighbors=5, weights='uniform', metric='euclidean')
    model.fit(X_train_f, y_train_f)
    y_pred = model.predict(X_test_f)
    f1_baseline.append(f1_score(y_test_f, y_pred, average='macro'))

print(f"Baseline mean macro-F1: {np.mean(f1_baseline):.4f} (± {np.std(f1_baseline):.4f})")

Baseline mean macro-F1: 0.3836 (± 0.0084)


In [ ]:
# ============================================================
# Cell C: Distance-weighted (k=5, Euclidean, all features, no SMOTE)
# ============================================================
f1_weighted = []
for X_train_f, X_test_f, y_train_f, y_test_f in fold_data_full:
    model = KNeighborsClassifier(n_neighbors=5, weights='distance', metric='euclidean')
    model.fit(X_train_f, y_train_f)
    y_pred = model.predict(X_test_f)
    f1_weighted.append(f1_score(y_test_f, y_pred, average='macro'))

print(f"Distance-weighted mean macro-F1: {np.mean(f1_weighted):.4f} (± {np.std(f1_weighted):.4f})")

Distance-weighted mean macro-F1: 0.3909 (± 0.0050)


In [ ]:
# ============================================================
# Cell D: Manhattan distance, k=7, weighted, all features, no SMOTE
# ============================================================
f1_manhattan = []
for X_train_f, X_test_f, y_train_f, y_test_f in fold_data_full:
    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train_f, y_train_f)
    y_pred = model.predict(X_test_f)
    f1_manhattan.append(f1_score(y_test_f, y_pred, average='macro'))

print(f"Manhattan k=7 mean macro-F1: {np.mean(f1_manhattan):.4f} (± {np.std(f1_manhattan):.4f})")

Manhattan k=7 mean macro-F1: 0.4151 (± 0.0027)


In [ ]:
# ============================================================
# Cell E: MI-selected top-10, Manhattan k=7, weighted, no SMOTE
# ============================================================
f1_mi_selected = []
for (X_train_f, X_test_f, y_train_f, y_test_f), top10 in zip(fold_data_full, fold_selected_features):
    X_train_top10 = X_train_f[top10]
    X_test_top10 = X_test_f[top10]

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train_top10, y_train_f)
    y_pred = model.predict(X_test_top10)
    f1_mi_selected.append(f1_score(y_test_f, y_pred, average='macro'))

print(f"MI-selected (no SMOTE) mean macro-F1: {np.mean(f1_mi_selected):.4f} (± {np.std(f1_mi_selected):.4f})")

MI-selected (no SMOTE) mean macro-F1: 0.4957 (± 0.0084)


In [ ]:
# ============================================================
# Cell F: Full corrected progression summary table
# ============================================================
print("Corrected, leak-safe progression:")
print(f"  Baseline (k=5, euclidean, uniform):        {np.mean(f1_baseline):.4f}")
print(f"  Distance-weighted:                          {np.mean(f1_weighted):.4f}")
print(f"  Manhattan, k=7:                              {np.mean(f1_manhattan):.4f}")
print(f"  MI-selected top-10 (no SMOTE):               {np.mean(f1_mi_selected):.4f}")
print(f"  Final (MI-selected + SMOTE capped at 5x):    {np.mean(fold_macro_f1):.4f}")

Corrected, leak-safe progression:
  Baseline (k=5, euclidean, uniform):        0.3836
  Distance-weighted:                          0.3909
  Manhattan, k=7:                              0.4151
  MI-selected top-10 (no SMOTE):               0.4957
  Final (MI-selected + SMOTE capped at 5x):    0.5043


In [ ]:
# ============================================================
# Quick re-check: feature count sweep on the corrected pipeline,
# single-fold screen (same pattern as the original test)
# ============================================================
X_train_f, X_test_f, y_train_f, y_test_f = fold_data_full[0]

# Recompute MI ranking for this fold (already have it from
# fold_selected_features[0], but need the FULL ranking, not
# just the top 10, to test larger N values)
mi_scores = mutual_info_classif(X_train_f, y_train_f, random_state=42)
mi_ranking = pd.Series(mi_scores, index=X_train_f.columns).sort_values(ascending=False)

for n in [10, 20, 30, 40, X_train_f.shape[1]]:
    top_n = mi_ranking.head(n).index.tolist()
    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train_f[top_n], y_train_f)
    y_pred = model.predict(X_test_f[top_n])
    macro_f1 = f1_score(y_test_f, y_pred, average='macro')
    print(f"top {n:3d} features: macro-F1 = {macro_f1:.4f}")

top  10 features: macro-F1 = 0.4947
top  20 features: macro-F1 = 0.4275
top  30 features: macro-F1 = 0.4235
top  40 features: macro-F1 = 0.4140
top  56 features: macro-F1 = 0.4156


In [ ]:
# ============================================================
# Sanity checks: no NaNs introduced, train/test columns aligned
# ============================================================
for i, (X_train_f, X_test_f, y_train_f, y_test_f) in enumerate(fold_data):
    assert X_train_f.isna().sum().sum() == 0, f"Fold {i+1}: NaNs in X_train"
    assert X_test_f.isna().sum().sum() == 0, f"Fold {i+1}: NaNs in X_test"
    assert list(X_train_f.columns) == list(X_test_f.columns), f"Fold {i+1}: column mismatch"

print("All folds clean: no NaNs, train/test columns aligned.")

All folds clean: no NaNs, train/test columns aligned.


In [ ]:
# ============================================================
# Confirm fold splits are reproducible (same seed = same splits
# every time skf.split is called on the same X, y)
# ============================================================
splits_1 = [test_idx.tolist() for _, test_idx in skf.split(X, y)]
splits_2 = [test_idx.tolist() for _, test_idx in skf.split(X, y)]
print("Fold splits reproducible:", splits_1 == splits_2)

Fold splits reproducible: True


In [ ]:
# ============================================================
# Re-check: min-max vs z-score on the CURRENT 10-feature set
# ============================================================
from sklearn.preprocessing import StandardScaler

X_train_f, X_test_f, y_train_f, y_test_f = fold_data[0]  # already MI-selected + SMOTE'd

# min-max version (current, already in fold_data)
model_mm = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
model_mm.fit(X_train_f, y_train_f)
y_pred_mm = model_mm.predict(X_test_f)
print(f"Min-max: macro-F1 = {f1_score(y_test_f, y_pred_mm, average='macro'):.4f}")

# z-score version, fit on this fold's SMOTE'd training data only
scaler_z = StandardScaler()
X_train_z = scaler_z.fit_transform(X_train_f)
X_test_z = scaler_z.transform(X_test_f)
model_z = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
model_z.fit(X_train_z, y_train_f)
y_pred_z = model_z.predict(X_test_z)
print(f"Z-score: macro-F1 = {f1_score(y_test_f, y_pred_z, average='macro'):.4f}")

Min-max: macro-F1 = 0.5020
Z-score: macro-F1 = 0.5012


In [ ]:
# ============================================================
# Cell G: Build the post-MI-selection, pre-resampling checkpoint
# — reused by every resampling comparison below
# ============================================================
fold_data_selected = []
for (X_train_f, X_test_f, y_train_f, y_test_f), top10 in zip(fold_data_full, fold_selected_features):
    fold_data_selected.append((X_train_f[top10], X_test_f[top10], y_train_f, y_test_f))

print("fold_data_selected built:", len(fold_data_selected), "folds")

fold_data_selected built: 5 folds


In [ ]:
# ============================================================
# Cell H: Flat-target SMOTE (target=5000) — the original
# configuration that failed on the old pipeline
# ============================================================
from imblearn.over_sampling import SMOTE

f1_smote_flat = []
for X_train_f, X_test_f, y_train_f, y_test_f in fold_data_selected:
    target_counts = {cls: max(count, 5000) for cls, count in y_train_f.value_counts().items()}
    smote = SMOTE(sampling_strategy=target_counts, random_state=42, k_neighbors=5)
    X_train_s, y_train_s = smote.fit_resample(X_train_f, y_train_f)

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train_s, y_train_s)
    y_pred = model.predict(X_test_f)
    f1_smote_flat.append(f1_score(y_test_f, y_pred, average='macro'))

print(f"SMOTE (flat target=5000): {np.mean(f1_smote_flat):.4f} (± {np.std(f1_smote_flat):.4f})")

SMOTE (flat target=5000): 0.4919 (± 0.0074)


In [ ]:
# ============================================================
# Cell I: Random oversampling (ratio-capped at 5x)
# ============================================================
from imblearn.over_sampling import RandomOverSampler

f1_ros = []
for X_train_f, X_test_f, y_train_f, y_test_f in fold_data_selected:
    target_counts = {cls: min(count * 5, y_train_f.value_counts().max())
                      for cls, count in y_train_f.value_counts().items()}
    ros = RandomOverSampler(sampling_strategy=target_counts, random_state=42)
    X_train_r, y_train_r = ros.fit_resample(X_train_f, y_train_f)

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train_r, y_train_r)
    y_pred = model.predict(X_test_f)
    f1_ros.append(f1_score(y_test_f, y_pred, average='macro'))

print(f"Random oversampling (5x capped): {np.mean(f1_ros):.4f} (± {np.std(f1_ros):.4f})")

Random oversampling (5x capped): 0.5004 (± 0.0050)


In [ ]:
# ============================================================
# Cell J: Tomek link removal alone (no oversampling)
# ============================================================
from imblearn.under_sampling import TomekLinks

f1_tomek = []
for X_train_f, X_test_f, y_train_f, y_test_f in fold_data_selected:
    tomek = TomekLinks()
    X_train_t, y_train_t = tomek.fit_resample(X_train_f, y_train_f)

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train_t, y_train_t)
    y_pred = model.predict(X_test_f)
    f1_tomek.append(f1_score(y_test_f, y_pred, average='macro'))

print(f"Tomek links alone: {np.mean(f1_tomek):.4f} (± {np.std(f1_tomek):.4f})")

Tomek links alone: 0.4917 (± 0.0081)


In [ ]:
# ============================================================
# Cell K: SMOTE (5x capped) + Tomek combined
# ============================================================
from imblearn.combine import SMOTETomek

f1_smote_tomek = []
for X_train_f, X_test_f, y_train_f, y_test_f in fold_data_selected:
    target_counts = {cls: min(count * 5, y_train_f.value_counts().max())
                      for cls, count in y_train_f.value_counts().items()}
    smote = SMOTE(sampling_strategy=target_counts, random_state=42, k_neighbors=5)
    smote_tomek = SMOTETomek(smote=smote, random_state=42)
    X_train_st, y_train_st = smote_tomek.fit_resample(X_train_f, y_train_f)

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train_st, y_train_st)
    y_pred = model.predict(X_test_f)
    f1_smote_tomek.append(f1_score(y_test_f, y_pred, average='macro'))

print(f"SMOTE (5x) + Tomek: {np.mean(f1_smote_tomek):.4f} (± {np.std(f1_smote_tomek):.4f})")

SMOTE (5x) + Tomek: 0.5060 (± 0.0118)


In [ ]:
# ============================================================
# Cell L: Full resampling comparison summary + paired t-test
# (SMOTE vs random oversampling, matching the original report claim)
# ============================================================
from scipy import stats

print("Resampling strategy comparison (leak-safe):")
print(f"  No resampling (MI-selected only):  {np.mean(f1_mi_selected):.4f}")
print(f"  SMOTE, flat target=5000:            {np.mean(f1_smote_flat):.4f}")
print(f"  SMOTE, ratio-capped 5x (adopted):   {np.mean(fold_macro_f1):.4f}")
print(f"  Random oversampling, 5x capped:      {np.mean(f1_ros):.4f}")
print(f"  Tomek links alone:                    {np.mean(f1_tomek):.4f}")
print(f"  SMOTE (5x) + Tomek:                   {np.mean(f1_smote_tomek):.4f}")

t_stat, p_val = stats.ttest_rel(fold_macro_f1, f1_ros)
print(f"\nPaired t-test, SMOTE vs random oversampling: t={t_stat:.4f}, p={p_val:.4f}")

Resampling strategy comparison (leak-safe):
  No resampling (MI-selected only):  0.4957
  SMOTE, flat target=5000:            0.4919
  SMOTE, ratio-capped 5x (adopted):   0.5043
  Random oversampling, 5x capped:      0.5004
  Tomek links alone:                    0.4917
  SMOTE (5x) + Tomek:                   0.5060

Paired t-test, SMOTE vs random oversampling: t=2.0075, p=0.1151


In [ ]:
# ============================================================
# Per-class check: does SMOTE+Tomek still hurt a specific class,
# same diagnostic pattern as before — never trust a higher mean
# without checking per-class impact first
# ============================================================
all_y_test_st = []
all_y_pred_st = []

for X_train_f, X_test_f, y_train_f, y_test_f in fold_data_selected:
    target_counts = {cls: min(count * 5, y_train_f.value_counts().max())
                      for cls, count in y_train_f.value_counts().items()}
    smote = SMOTE(sampling_strategy=target_counts, random_state=42, k_neighbors=5)
    smote_tomek = SMOTETomek(smote=smote, random_state=42)
    X_train_st, y_train_st = smote_tomek.fit_resample(X_train_f, y_train_f)

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(X_train_st, y_train_st)
    y_pred = model.predict(X_test_f)

    all_y_test_st.extend(y_test_f)
    all_y_pred_st.extend(y_pred)

print("SMOTE (5x) + Tomek, pooled per-class report:")
print(classification_report(all_y_test_st, all_y_pred_st, digits=3))

SMOTE (5x) + Tomek, pooled per-class report:
              precision    recall  f1-score   support

           0      0.926     0.827     0.874     83358
           1      0.696     0.729     0.712      8609
           2      0.201     0.368     0.260      1503
           3      0.290     0.427     0.346      5019
           4      0.767     0.758     0.762     26927
           5      0.139     0.091     0.110      1577
           6      0.497     0.648     0.563     19470
           7      0.399     0.562     0.467       169
           8      0.346     0.377     0.361      1426
           9      0.703     0.566     0.627      2185

    accuracy                          0.752    150243
   macro avg      0.496     0.535     0.508    150243
weighted avg      0.782     0.752     0.764    150243



In [ ]:
# ============================================================
# Statistical test: is SMOTE+Tomek's improvement over plain
# SMOTE actually significant, or within noise (same rigor as
# the SMOTE vs ROS test already in the report)
# ============================================================
t_stat_st, p_val_st = stats.ttest_rel(f1_smote_tomek, fold_macro_f1)
print(f"Paired t-test, SMOTE+Tomek vs SMOTE alone: t={t_stat_st:.4f}, p={p_val_st:.4f}")

Paired t-test, SMOTE+Tomek vs SMOTE alone: t=0.3094, p=0.7725


In [ ]:
# ============================================================
# Cell M: Full k/metric grid, properly verified on the
# corrected, leak-safe pipeline (closes the previously
# flagged unverified claim)
# ============================================================
k_values = [1, 3, 5, 7, 9, 15, 21, 31]
metrics = ['euclidean', 'manhattan']

X_train_f, X_test_f, y_train_f, y_test_f = fold_data_full[0]  # single-fold screen, same pattern as before

results = []
for metric in metrics:
    for k in k_values:
        model = KNeighborsClassifier(n_neighbors=k, weights='distance', metric=metric)
        model.fit(X_train_f, y_train_f)
        y_pred = model.predict(X_test_f)
        macro_f1 = f1_score(y_test_f, y_pred, average='macro')
        results.append({'k': k, 'metric': metric, 'macro_f1': macro_f1})
        print(f"metric={metric:10s} k={k:3d}  macro-F1={macro_f1:.4f}")

results_df = pd.DataFrame(results)
best = results_df.loc[results_df['macro_f1'].idxmax()]
print(f"\nBest: metric={best['metric']}, k={best['k']}, macro-F1={best['macro_f1']:.4f}")

metric=euclidean  k=  1  macro-F1=0.3763
metric=euclidean  k=  3  macro-F1=0.3811
metric=euclidean  k=  5  macro-F1=0.3930
metric=euclidean  k=  7  macro-F1=0.3903
metric=euclidean  k=  9  macro-F1=0.3912
metric=euclidean  k= 15  macro-F1=0.3918
metric=euclidean  k= 21  macro-F1=0.3877
metric=euclidean  k= 31  macro-F1=0.3754
metric=manhattan  k=  1  macro-F1=0.3945
metric=manhattan  k=  3  macro-F1=0.4043
metric=manhattan  k=  5  macro-F1=0.4174
metric=manhattan  k=  7  macro-F1=0.4156
metric=manhattan  k=  9  macro-F1=0.4135
metric=manhattan  k= 15  macro-F1=0.4039
metric=manhattan  k= 21  macro-F1=0.3927
metric=manhattan  k= 31  macro-F1=0.3797

Best: metric=manhattan, k=5, macro-F1=0.4174


In [ ]:
# ============================================================
# Cell N: Re-check min-max vs z-score against the NEW final
# configuration (SMOTE + Tomek), not plain SMOTE
# ============================================================
X_train_f, X_test_f, y_train_f, y_test_f = fold_data_selected[0]

target_counts = {cls: min(count * 5, y_train_f.value_counts().max())
                  for cls, count in y_train_f.value_counts().items()}
smote = SMOTE(sampling_strategy=target_counts, random_state=42, k_neighbors=5)
smote_tomek = SMOTETomek(smote=smote, random_state=42)
X_train_st, y_train_st = smote_tomek.fit_resample(X_train_f, y_train_f)

# min-max (current)
model_mm = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
model_mm.fit(X_train_st, y_train_st)
y_pred_mm = model_mm.predict(X_test_f)
print(f"Min-max: macro-F1 = {f1_score(y_test_f, y_pred_mm, average='macro'):.4f}")

# z-score
scaler_z = StandardScaler()
X_train_stz = scaler_z.fit_transform(X_train_st)
X_test_z = scaler_z.transform(X_test_f)
model_z = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
model_z.fit(X_train_stz, y_train_st)
y_pred_z = model_z.predict(X_test_z)
print(f"Z-score: macro-F1 = {f1_score(y_test_f, y_pred_z, average='macro'):.4f}")

Min-max: macro-F1 = 0.5222
Z-score: macro-F1 = 0.5153


In [ ]:
# ============================================================
# Re-confirm resampling comparison at k=5 (was k=7)
# ============================================================
f1_smote_flat_k5, f1_smote_k5, f1_ros_k5, f1_tomek_k5, f1_smote_tomek_k5 = [], [], [], [], []

for X_train_f, X_test_f, y_train_f, y_test_f in fold_data_selected:
    # Flat-target SMOTE
    target_flat = {cls: max(count, 5000) for cls, count in y_train_f.value_counts().items()}
    smote_flat = SMOTE(sampling_strategy=target_flat, random_state=42, k_neighbors=5)
    X_s, y_s = smote_flat.fit_resample(X_train_f, y_train_f)
    m = KNeighborsClassifier(n_neighbors=5, weights='distance', metric='manhattan')
    m.fit(X_s, y_s); f1_smote_flat_k5.append(f1_score(y_test_f, m.predict(X_test_f), average='macro'))

    # Ratio-capped SMOTE
    target_capped = {cls: min(count * 5, y_train_f.value_counts().max()) for cls, count in y_train_f.value_counts().items()}
    smote_capped = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5)
    X_s, y_s = smote_capped.fit_resample(X_train_f, y_train_f)
    m = KNeighborsClassifier(n_neighbors=5, weights='distance', metric='manhattan')
    m.fit(X_s, y_s); f1_smote_k5.append(f1_score(y_test_f, m.predict(X_test_f), average='macro'))

    # Random oversampling
    ros = RandomOverSampler(sampling_strategy=target_capped, random_state=42)
    X_r, y_r = ros.fit_resample(X_train_f, y_train_f)
    m = KNeighborsClassifier(n_neighbors=5, weights='distance', metric='manhattan')
    m.fit(X_r, y_r); f1_ros_k5.append(f1_score(y_test_f, m.predict(X_test_f), average='macro'))

    # Tomek alone
    tomek = TomekLinks()
    X_t, y_t = tomek.fit_resample(X_train_f, y_train_f)
    m = KNeighborsClassifier(n_neighbors=5, weights='distance', metric='manhattan')
    m.fit(X_t, y_t); f1_tomek_k5.append(f1_score(y_test_f, m.predict(X_test_f), average='macro'))

    # SMOTE + Tomek
    smote_st = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5)
    smote_tomek = SMOTETomek(smote=smote_st, random_state=42)
    X_st, y_st = smote_tomek.fit_resample(X_train_f, y_train_f)
    m = KNeighborsClassifier(n_neighbors=5, weights='distance', metric='manhattan')
    m.fit(X_st, y_st); f1_smote_tomek_k5.append(f1_score(y_test_f, m.predict(X_test_f), average='macro'))

print(f"SMOTE flat:          {np.mean(f1_smote_flat_k5):.4f} (± {np.std(f1_smote_flat_k5):.4f})")
print(f"SMOTE ratio-capped:  {np.mean(f1_smote_k5):.4f} (± {np.std(f1_smote_k5):.4f})")
print(f"Random oversampling: {np.mean(f1_ros_k5):.4f} (± {np.std(f1_ros_k5):.4f})")
print(f"Tomek alone:         {np.mean(f1_tomek_k5):.4f} (± {np.std(f1_tomek_k5):.4f})")
print(f"SMOTE + Tomek:       {np.mean(f1_smote_tomek_k5):.4f} (± {np.std(f1_smote_tomek_k5):.4f})")

t_stat, p_val = stats.ttest_rel(f1_smote_tomek_k5, f1_smote_k5)
print(f"\nPaired t-test, SMOTE+Tomek vs SMOTE alone: t={t_stat:.4f}, p={p_val:.4f}")

SMOTE flat:          0.4863 (± 0.0056)
SMOTE ratio-capped:  0.5001 (± 0.0077)
Random oversampling: 0.4967 (± 0.0089)
Tomek alone:         0.4916 (± 0.0092)
SMOTE + Tomek:       0.4992 (± 0.0104)

Paired t-test, SMOTE+Tomek vs SMOTE alone: t=-0.1761, p=0.8688


In [ ]:
# ============================================================
# Re-confirm min-max vs z-score at k=5, against SMOTE+Tomek
# ============================================================
X_train_f, X_test_f, y_train_f, y_test_f = fold_data_selected[0]
target_capped = {cls: min(count * 5, y_train_f.value_counts().max()) for cls, count in y_train_f.value_counts().items()}
smote_st = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5)
smote_tomek = SMOTETomek(smote=smote_st, random_state=42)
X_train_st, y_train_st = smote_tomek.fit_resample(X_train_f, y_train_f)

model_mm = KNeighborsClassifier(n_neighbors=5, weights='distance', metric='manhattan')
model_mm.fit(X_train_st, y_train_st)
print(f"Min-max: macro-F1 = {f1_score(y_test_f, model_mm.predict(X_test_f), average='macro'):.4f}")

scaler_z = StandardScaler()
X_train_stz = scaler_z.fit_transform(X_train_st)
X_test_z = scaler_z.transform(X_test_f)
model_z = KNeighborsClassifier(n_neighbors=5, weights='distance', metric='manhattan')
model_z.fit(X_train_stz, y_train_st)
print(f"Z-score: macro-F1 = {f1_score(y_test_f, model_z.predict(X_test_z), average='macro'):.4f}")

Min-max: macro-F1 = 0.5095
Z-score: macro-F1 = 0.5077
